# Perseptron v2 - 00 Customer-Level Fold Hazirligi

Bu notebook proposal'da istenen **customer-level 5-fold validation** duzenini hazirlar. Amac, ayni musterinin hem train hem validation tarafinda yer almasini engellemek ve daha savunulabilir bir deney protokolu kurmaktir.

Cikti dosyalari:

- `reports/proposal_v2/proposal_v2_fold_splits.csv`
- `reports/proposal_v2/proposal_v2_fold_protocol_summary.json`

## 1. Ortam ve Proje Yolu

Kaggle'da bu branch kodlari notebook inputu olarak eklenmis olabilir veya `/kaggle/working` altinda bulunabilir. Asagidaki hucre proje klasorunu otomatik bulur.

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

def find_source_project():
    candidates = [Path('/kaggle/working/perseptron_project'), Path.cwd(), Path('/kaggle/input/perseptron-project')]
    candidates += [path for path in Path('/kaggle/input').glob('*') if path.is_dir()]
    for path in candidates:
        if (path / 'src' / 'proposal_v2').exists():
            return path
    return Path.cwd()

SOURCE_PROJECT_DIR = find_source_project()
WORK_PROJECT_DIR = Path('/kaggle/working/perseptron_project_work') if Path('/kaggle').exists() else SOURCE_PROJECT_DIR
if SOURCE_PROJECT_DIR != WORK_PROJECT_DIR:
    shutil.copytree(SOURCE_PROJECT_DIR, WORK_PROJECT_DIR, dirs_exist_ok=True)

PROJECT_DIR = WORK_PROJECT_DIR
os.environ['PERSEPTRON_PROJECT_DIR'] = str(PROJECT_DIR)
os.environ['PYTHONPATH'] = str(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))
print('SOURCE_PROJECT_DIR =', SOURCE_PROJECT_DIR)
print('PROJECT_DIR =', PROJECT_DIR)

def restore_previous_outputs():
    if not Path('/kaggle/input').exists():
        return
    for input_root in Path('/kaggle/input').glob('*'):
        if input_root == SOURCE_PROJECT_DIR:
            continue
        for relative in ['reports/proposal_v2', 'models/proposal_v2']:
            source = input_root / relative
            target = PROJECT_DIR / relative
            if source.exists():
                target.mkdir(parents=True, exist_ok=True)
                shutil.copytree(source, target, dirs_exist_ok=True)
                print('Restored', source, '->', target)

restore_previous_outputs()

def run_module(module, *args):
    command = [sys.executable, '-m', module, *map(str, args)]
    print('RUN:', ' '.join(command))
    subprocess.run(command, cwd=PROJECT_DIR, check=True)


## 2. Parametreler

Proposal uyumu icin `N_FOLDS = 5` sabit tutulur. `MIN_HISTORY`, validasyon icin yeterli gecmisi olmayan musterileri elemek icindir.

In [ ]:
N_FOLDS = 5
RANDOM_SEED = 42
MIN_HISTORY = 2
VALIDATION_DAYS = 7


## 3. Fold Dosyalarini Uret

Bu adim train/validation customer intersection degerinin her fold icin `0` oldugunu kontrol eder. Hata verirse sonraki notebooklara gecilmemelidir.

In [ ]:
run_module(
    'src.proposal_v2.splits',
    '--n-folds', N_FOLDS,
    '--seed', RANDOM_SEED,
    '--min-history', MIN_HISTORY,
    '--validation-days', VALIDATION_DAYS,
)
